# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline Rule

- A content page should be prioritized for refresh when it is old, still receives meaningfull search visibility, and shows signs of deciling search performance.
- The rule intenionally uses only information present at the time of decision and does not rely on future outcomes or label-derived features.
- The resulting score is a transparent baseline that will later can be compared with machine learning model.

### Reason Code

The baseline rule assigns one reason code to every page that is recommended for review.

| Reason Code | Meaning | Supported Action |
|-------------|---------|------------------|
| `STALE_VISIBLE_DECLINING` | The content page is old, still receives meaningful search visibility, and shows signs of declining performance. These combined signals suggest that refreshing the content may provide value. | `REFRESH_REVIEW` |

The reason code is attached to every recommendation so that a human reviewer understands **why** the page was selected. It makes the ranked queue transparent and easy to audit. The recommendation is intended to support editorial review rather than automatically refreshing the content. 

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import numpy as np
from pathlib import Path
import pandas as pd

repo_root = Path.cwd().parents[1]

data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print(f"Loaded: {data_path}")

df.head()

Loaded: c:\flyrank-ml-internship\data\raw\content_refresh_anonymized.csv


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  str    
 1   client_id               30000 non-null  str    
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  str    
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  str    
 7   main_intent             27626 non-null  str    
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   str    
 11  model_used              24267 non-null  str    
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null  int64  
 

In [5]:
required_columns = [
    "days_since_last_update",
    "impressions_90d",
    "search_volume",
    "content_type",
    "ctr",
    "avg_position",
    "trend_direction",
    "trend_pct"
]

print(df[required_columns].head())

   days_since_last_update  impressions_90d  search_volume     content_type  \
0                      20             3803           10.0  keyword article   
1                      25            15320           90.0  keyword article   
2                      20            12581            0.0  keyword article   
3                      22            11751           10.0  keyword article   
4                      14            19140            0.0  keyword article   

    ctr  avg_position trend_direction  trend_pct  
0  0.76          10.6            down      -41.4  
1  0.05          20.3            down      -57.7  
2  0.09          36.5            down      -60.9  
3  0.49           6.2          stable      -13.8  
4  0.13          44.0            down      -34.7  


### Signal Check 1 — Staleness

Hypothesis:

Older content is more likely to require a refresh than recently updated content.

To test this, pages are grouped into update-age buckets, and the average trend percentage together with the number of pages in each bucket is examined.

In [6]:
stale_bucket = pd.cut(
    df["days_since_last_update"],
    bins=[0, 90, 180, 365, np.inf],
    labels=["0-90", "91-180", "181-365", "365+"]
)

signal1 = (
    df.groupby(stale_bucket)
      .agg(
          n=("content_id", "count"),
          avg_trend=("trend_pct", "mean")
      )
      .reset_index()
)

signal1

,days_since_last_update,n,avg_trend
0,0-90,20655,0.707094
1,91-180,9171,-15.683224
2,181-365,169,-4.718462
3,365+,5,-96.166667


**Verdict:** CONFIRMED

Older pages generally show weaker trend performance than recently updated pages, supporting the use of content staleness as one component of the baseline refresh rule.

### Signal Check 2 — Visibility

Hypothesis:

Pages with higher search visibility provide larger opportunities for content refresh because improvements can influence more users.

In [7]:
visibility_bucket = pd.cut(
    df["impressions_90d"],
    bins=[-1, 100, 500, 1000, np.inf],
    labels=["0-100", "101-500", "501-1000", "1000+"]
)

signal2 = (
    df.groupby(visibility_bucket)
      .agg(
          n=("content_id", "count"),
          avg_search_volume=("search_volume", "mean")
      )
      .reset_index()
)

signal2

,impressions_90d,n,avg_search_volume
0,0-100,8006,135.039344
1,101-500,5279,157.101304
2,501-1000,3206,167.745665
3,1000+,13509,168.386710


**Verdict:** CONFIRMED

Higher-impression pages generally correspond to higher search opportunities, making visibility an appropriate signal for prioritizing refresh candidates.

### Build the Baseline Score

The baseline rule is converted into a transparent numerical score using simple, human-readable conditions. No machine learning is used. Each condition contributes a fixed number of points so that the final score remains easy to understand and explain.

Only information available at the decision time is used. Label-derived fields such as `trend_direction` and `trend_pct` are deliberately excluded to avoid data leakage.

In [8]:
# Signal 1: Content is stale
stale = (df["days_since_last_update"] >= 180).astype(int)

# Signal 2: Page still receives meaningful search visibility
visible = (df["impressions_90d"] >= 500).astype(int)

# Signal 3: High search opportunity
high_value = (df["search_volume"] >= 500).fillna(False).astype(int)

df["baseline_score"] = (
    stale * 3 +
    visible * 2 +
    high_value * 1
)

# Preview the score
df[[
    "content_id",
    "days_since_last_update",
    "impressions_90d",
    "search_volume",
    "baseline_score"
]].head()

,content_id,days_since_last_update,impressions_90d,search_volume,baseline_score
0,content_304f48230142,20,3803,10.0,2
1,content_a1fb4e703a9e,25,15320,90.0,2
2,content_9aa793d4d895,20,12581,0.0,2
3,content_331d6c4de07b,22,11751,10.0,2
4,content_d99b7a2d90ca,14,19140,0.0,2


### Assign Reason Codes

Each page receives a reason code explaining why it was prioritized. The reason code makes the ranked queue transparent so that a reviewer understands which observable signals contributed to the recommendation.

For this baseline, pages that satisfy the refresh rule receive the reason code `STALE_VISIBLE_HIGHVALUE`. Pages that do not meet the rule receive `LOW_PRIORITY`.

In [9]:
def assign_reason(row):
    if row["baseline_score"] >= 5:
        return "STALE_VISIBLE_HIGHVALUE"
    else:
        return "LOW_PRIORITY"

df["reason_code"] = df.apply(assign_reason, axis=1)

df[["content_id", "baseline_score", "reason_code"]].head()

,content_id,baseline_score,reason_code
0,content_304f48230142,2,LOW_PRIORITY
1,content_a1fb4e703a9e,2,LOW_PRIORITY
2,content_9aa793d4d895,2,LOW_PRIORITY
3,content_331d6c4de07b,2,LOW_PRIORITY
4,content_d99b7a2d90ca,2,LOW_PRIORITY


### Assign Action Labels

The baseline converts each score into an action recommendation. High-scoring pages are sent for refresh review, while lower-scoring pages remain under monitoring.

The action label supports editorial decision-making and does not automatically modify any content.

In [10]:
def assign_action(row):
    if row["baseline_score"] >= 3:
        return "REFRESH_REVIEW"
    else:
        return "MONITOR"

df["action_label"] = df.apply(assign_action, axis=1)

df[["content_id", "baseline_score", "action_label"]].sample(5)

,content_id,baseline_score,action_label
21082,content_0e3dfa2a2766,1,MONITOR
27825,content_5634f9b47153,0,MONITOR
297,content_2ece38fb001c,2,MONITOR
21844,content_4fce48585ec2,2,MONITOR
17492,content_cfe0815a3229,2,MONITOR


### Build the Ranked Queue

The dataset is sorted in descending order of the baseline score so that the highest-priority pages appear first. This ranked queue represents the baseline recommendation list that later machine learning models will be compared against.

In [11]:
ranked_df = (
    df.sort_values(
        by="baseline_score",
        ascending=False
    )
    .reset_index(drop=True)
)

ranked_df.head(10)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code,action_label
0,content_b16bd7307b39,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4329.0,27844.0,...,0.00,25.00,0.0,good,page_3_5,down,-69.7,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW
1,content_72496874f806,client_4ec9599fc2,NaN,NaN,NaN,NaN,keyword article,NaN,1504.0,10770.0,...,0.00,0.00,0.0,moderate,page_1,down,-22.4,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW
2,content_fe16a55cd13d,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3388.0,21742.0,...,2.38,38.64,0.0,good,striking,down,-52.2,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW
3,content_1bfaa38ff26c,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3861.0,24672.0,...,3.75,43.33,0.0,good,page_3_5,down,-74.7,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW
4,content_074ba6ead17b,client_d029fa3a95,0.0,0.0,LOW,0.0,keyword article,informational,3994.0,27901.0,...,0.00,50.00,0.0,moderate,page_3_5,down,-36.5,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW
5,content_7f116ae1f6f5,client_9400f1b21c,NaN,NaN,NaN,NaN,keyword article,NaN,1335.0,9375.0,...,0.00,0.00,0.0,moderate,page_1,down,-44.5,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW
6,content_ecb6215e79fd,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4486.0,29333.0,...,25.00,33.33,0.0,good,page_3_5,down,-74.4,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW
7,content_77d4d5930e5e,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4020.0,26513.0,...,50.00,20.00,0.0,moderate,striking,down,-55.0,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW
8,content_bdbec75c1148,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3696.0,24643.0,...,0.00,33.33,0.0,moderate,page_3_5,stable,-11.7,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW
9,content_e3ff1b093148,client_d029fa3a95,0.0,0.0,LOW,0.0,keyword article,informational,4758.0,33575.0,...,0.00,20.00,0.0,moderate,page_1,down,-68.5,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW


In [13]:
import os
output_dir = "../outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "baseline_action_score.csv")

ranked_df.to_csv(output_path, index=False)

print(f"Baseline queue saved to: {output_path}")

Baseline queue saved to: ../outputs\baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

The highest-ranked pages are manually reviewed to determine whether the baseline recommendations appear reasonable. For each page, the assigned action, reason code, confidence, and a possible failure case are recorded.

This review helps identify weaknesses in the baseline rule before comparing it with a machine learning model.

In [14]:
top20 = ranked_df.head(20).copy()

top20

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code,action_label
0,content_b16bd7307b39,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,4329.0,27844.0,...,0.00,25.00,0.0,good,page_3_5,down,-69.7,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW
1,content_72496874f806,client_4ec9599fc2,NaN,NaN,NaN,NaN,keyword article,NaN,1504.0,10770.0,...,0.00,0.00,0.0,moderate,page_1,down,-22.4,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW
2,content_fe16a55cd13d,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3388.0,21742.0,...,2.38,38.64,0.0,good,striking,down,-52.2,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW
3,content_1bfaa38ff26c,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3861.0,24672.0,...,3.75,43.33,0.0,good,page_3_5,down,-74.7,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW
4,content_074ba6ead17b,client_d029fa3a95,0.0,0.00,LOW,0.00,keyword article,informational,3994.0,27901.0,...,0.00,50.00,0.0,moderate,page_3_5,down,-36.5,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW
5,content_7f116ae1f6f5,client_9400f1b21c,NaN,NaN,NaN,NaN,keyword article,NaN,1335.0,9375.0,...,0.00,0.00,0.0,moderate,page_1,down,-44.5,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW
6,content_ecb6215e79fd,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,4486.0,29333.0,...,25.00,33.33,0.0,good,page_3_5,down,-74.4,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW
7,content_77d4d5930e5e,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,4020.0,26513.0,...,50.00,20.00,0.0,moderate,striking,down,-55.0,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW
8,content_bdbec75c1148,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3696.0,24643.0,...,0.00,33.33,0.0,moderate,page_3_5,stable,-11.7,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW
9,content_e3ff1b093148,client_d029fa3a95,0.0,0.00,LOW,0.00,keyword article,informational,4758.0,33575.0,...,0.00,20.00,0.0,moderate,page_1,down,-68.5,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW


In [15]:
review = top20[[
    "content_id",
    "baseline_score",
    "reason_code",
    "action_label"
]].copy()

def confidence(score):
    if score >= 5:
        return "High Confidence"
    elif score >= 3:
        return "Medium Confidence"
    else:
        return "Low Confidence"

review["confidence_note"] = review["baseline_score"].apply(confidence)
review["what_would_make_it_wrong"] = (
    "The page may have been refreshed recently, be affected by seasonality, "
    "or experience temporary ranking fluctuations not captured by the baseline."
)

review

,content_id,baseline_score,reason_code,action_label,confidence_note,what_would_make_it_wrong
0,content_b16bd7307b39,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW,High Confidence,"The page may have been refreshed recently, be ..."
1,content_72496874f806,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW,High Confidence,"The page may have been refreshed recently, be ..."
2,content_fe16a55cd13d,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW,High Confidence,"The page may have been refreshed recently, be ..."
3,content_1bfaa38ff26c,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW,High Confidence,"The page may have been refreshed recently, be ..."
4,content_074ba6ead17b,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW,High Confidence,"The page may have been refreshed recently, be ..."
5,content_7f116ae1f6f5,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW,High Confidence,"The page may have been refreshed recently, be ..."
6,content_ecb6215e79fd,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW,High Confidence,"The page may have been refreshed recently, be ..."
7,content_77d4d5930e5e,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW,High Confidence,"The page may have been refreshed recently, be ..."
8,content_bdbec75c1148,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW,High Confidence,"The page may have been refreshed recently, be ..."
9,content_e3ff1b093148,5,STALE_VISIBLE_HIGHVALUE,REFRESH_REVIEW,High Confidence,"The page may have been refreshed recently, be ..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Although the baseline rule is transparent and easy to explain, it cannot capture every real-world situation. Some highly ranked pages may not actually require a refresh because the rule only considers a small number of observable signals.

Possible situations where the recommendation could be incorrect are listed below.

In [16]:
ranked_df.tail(5)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code,action_label
29995,content_2da6ae9d0882,client_e629fa6598,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.0,25.0,0.0,low,striking,down,-100.0,0,LOW_PRIORITY,MONITOR
29996,content_9d548144b06d,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3455.0,21571.0,...,0.0,0.0,0.0,low,striking,stable,11.1,0,LOW_PRIORITY,MONITOR
29997,content_af865035b328,client_f369cb89fc,30.0,1.00,HIGH,1.57,keyword article,transactional,2673.0,16168.0,...,0.0,50.0,0.0,low,page_1,down,-67.7,0,LOW_PRIORITY,MONITOR
29998,content_40cb4af260c0,client_f369cb89fc,0.0,0.00,LOW,0.00,keyword article,informational,3086.0,23347.0,...,0.0,50.0,0.0,low,page_3_5,down,-83.3,0,LOW_PRIORITY,MONITOR
29999,content_9bd30342fd4a,client_19581e27de,70.0,0.95,HIGH,0.34,keyword article,transactional,NaN,NaN,...,0.0,0.0,0.0,moderate,page_3_5,stable,0.0,0,LOW_PRIORITY,MONITOR


### Examples of Weak Picks

- A page may already have been refreshed recently, but the dataset has not yet reflected the improvement.

- A temporary seasonal decline may make an otherwise healthy page appear to be a refresh candidate.

- Some pages naturally receive low traffic because of niche topics rather than poor content quality.

- High search volume alone does not guarantee that refreshing the page will improve performance.

- Editorial priorities, business goals, or recent manual changes are not represented in the dataset and therefore cannot influence the baseline score.

## Leakage Check

The baseline score was intentionally built using only information that would be available when an editor decides whether a page should be reviewed.

The following checks confirm that no future information or product-generated recommendation flags were used.

In [17]:
used_features = [
    "days_since_last_update",
    "impressions_90d",
    "search_volume"
]

print("Features used in the baseline:")
for feature in used_features:
    print(f"- {feature}")

Features used in the baseline:
- days_since_last_update
- impressions_90d
- search_volume


### Leakage Verification

The baseline does **not** use:

- `trend_direction`
- `trend_pct`
- any future performance metrics
- any product-generated recommendation flags

All scoring features are available before the refresh decision is made, making the baseline suitable for honest evaluation.

Because the baseline relies only on decision-time information, it can be fairly compared against future machine learning models without introducing information leakage.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.